# Female Candidates

In [1]:
from importlib.metadata import version
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys

from scientisttools import FAMD, MCA
# Gower distance + UMAP — nonlinear mixed‑datatype embedding
import gower

# HDBSCAN — the best default for UMAP embeddings
import hdbscan
# Agglomerative (Hierarchical) Clustering
from sklearn.cluster import AgglomerativeClustering
from sklearn_extra.cluster import KMedoids

from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.cluster.hierarchy import linkage
from scipy.spatial.distance import squareform

# python source path
sys.path.append('../Src/')
# Set seed
SEED = 1776

# custom python
import plot
import utilities as u

# Create a dictionary of versions
versions = {
    "Python": sys.version.split()[0],
    "Pandas": pd.__version__,
    "NumPy": np.__version__,
    "Seaborn": sns.__version__,
    "Matplot": sys.modules['matplotlib'].__version__,
    "Sk-Learn": sys.modules['sklearn'].__version__,
    "Scipy": sys.modules['scipy'].__version__,
    "sklearn_extra": sys.modules['sklearn_extra'].__version__,
    "HDB Scan": version("hdbscan"),
    "Scientisttools": version("scientisttools"),
    "Gower": version("gower"),
}

# Display as a clean DataFrame
df_versions = pd.DataFrame(list(versions.items()), columns=['Library', 'Version'])
print(df_versions)

# display all columns
# pd.set_option('display.max_columns', None)
# pd.set_option('display.max_rows', None)
# pd.set_option('display.max_colwidth', None)

# Remove scientific notation
np.set_printoptions(suppress=True, precision=4, linewidth=100)
# reset options
# pd.reset_option('display.max_columns')

           Library  Version
0           Python  3.10.20
1           Pandas    2.3.3
2            NumPy   1.26.4
3          Seaborn   0.13.2
4          Matplot   3.10.8
5         Sk-Learn    1.7.2
6            Scipy   1.15.3
7    sklearn_extra    0.3.0
8         HDB Scan   0.8.42
9   Scientisttools    0.1.6
10           Gower    0.1.2


## Import Data

In [2]:
# import data
df = pd.read_parquet("../Data/Female_CAN_Heart_PreML.parquet")
mapping_df = pd.read_parquet("../Data/Female_CAN_Heart_Mapping.parquet")
# display shape
df.shape, mapping_df.shape

((8064, 104), (78, 18))

In [3]:
# check for NaNs
u.any_nans(df)

Clean Dataset: No NaNs found across 8,064 rows.


In [4]:
# display
df.head()

,PreviousTransplantNumber_CAN,WaitListDiagnosisCode_CAN,Citizenship_CAN,ResidencyStateRegistration_CAN,EducationLevel_CAN,LifeSupportRegistration_ECMO_CAN,LifeSupportRegistration_IABP_CAN,LifeSupportMechanismRegistration_OTHER_CAN,VentricularDeviceBrandRegistration_CAN,FunctionalStatusRegistration_CAN,...,FunctionalStatusTransplant_IsMissing,CreatinineRegistration_Log,CreatinineRegistration_Log_IsMissing,CreatinineTransplant_Log,CreatinineTransplant_Log_IsMissing,TotalBilirubinTransplant_Log,TotalBilirubinTransplant_Log_IsMissing,PC1_BMI_CAN,PC1_weight_CAN,PC1_height_CAN
0,Group_1,Group_2,Group_2,Group_1,Group_1,Group_1,Group_1,Group_1,Unknown,Group_3,...,0,0.832909,0,0.832909,0,0.405465,0,3.742386,2.043337,-2.418191
1,Group_1,Group_2,Group_2,Group_1,Group_1,Group_1,Group_1,Group_1,Unknown,Group_4,...,0,0.693147,0,0.693147,0,0.470004,0,-1.218457,-2.587516,-4.307358
2,Group_1,Group_2,Other,Group_1,Group_1,Group_1,Group_2,Group_1,CentriMag (Thoratec/Levitronix),Group_4,...,0,0.470004,0,0.536493,0,0.262364,0,-0.981478,-1.075258,-0.445072
3,Group_1,Group_2,Group_2,Group_1,Group_1,Group_1,Group_1,Group_1,Unknown,Group_4,...,0,0.737164,0,0.717840,0,0.336472,0,2.481018,1.531014,-1.396741
4,Group_1,Group_2,Group_2,Group_2,Group_1,Group_1,Group_2,Group_1,Unknown,Group_3,...,0,0.693147,0,0.916291,0,0.875469,0,-0.575276,-0.345262,0.498493


### User Function(s)

In [5]:
def display_mapping(mapping_data, feature):
    return mapping_data.loc[mapping_data.feature == feature].dropna(axis=1, how='all')

## Explortory Clustering

In [6]:
df.info(max_cols=df.shape[1])

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8064 entries, 0 to 8063
Data columns (total 104 columns):
 #    Column                                        Non-Null Count  Dtype   
---   ------                                        --------------  -----   
 0    PreviousTransplantNumber_CAN                  8064 non-null   category
 1    WaitListDiagnosisCode_CAN                     8064 non-null   category
 2    Citizenship_CAN                               8064 non-null   category
 3    ResidencyStateRegistration_CAN                8064 non-null   category
 4    EducationLevel_CAN                            8064 non-null   category
 5    LifeSupportRegistration_ECMO_CAN              8064 non-null   category
 6    LifeSupportRegistration_IABP_CAN              8064 non-null   category
 7    LifeSupportMechanismRegistration_OTHER_CAN    8064 non-null   category
 8    VentricularDeviceBrandRegistration_CAN        8064 non-null   category
 9    FunctionalStatusRegistration_CAN       

In [7]:
# get columns
shadow_cols = df.columns[df.columns.str.contains('IsMissing')].tolist()
pca_cols =  df.columns[df.columns.str.contains('^PC')].tolist()
cat_cols = df.select_dtypes(include=["category"]).columns.tolist()
# all numeric cols & remove numeric columns above
num_cols = df.select_dtypes(include=["number"]).columns.tolist()
num_cols = list(set(num_cols) - set(shadow_cols) - set(pca_cols))
# sanity check
print(f"Shadow Columns: {len(shadow_cols)} PCA Columns: {len(pca_cols)} Numeric Columns: {len(num_cols)} Category Columns: {len(cat_cols)}")
print(f"Total Columns: {len(shadow_cols)+len(pca_cols)+len(cat_cols)+len(num_cols)} Data Frame Columns: {df.shape[1]}")

Shadow Columns: 6 PCA Columns: 9 Numeric Columns: 8 Category Columns: 81
Total Columns: 104 Data Frame Columns: 104


## Data Wrangle

In [8]:
# remove label
df = df.drop('TransplantSurvivalDay', axis=1).copy()